# Day S4 — 쌍 정렬과 동적 프로그래밍: 표를 채워 최적 정렬 찾기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/supplement/day-S4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/supplement/day-S4.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 글자로 그리는 dot plot

In [ ]:
def dot_plot(seq_a, seq_b, window=1):
    """seq_a를 세로, seq_b를 가로에 놓고 window칸이 연속으로 같으면 점을 찍는다."""
    print("  " + seq_b)
    for i in range(len(seq_a)):
        row = ""
        for j in range(len(seq_b)):
            same = seq_a[i:i + window] == seq_b[j:j + window] and i + window <= len(seq_a)
            row += "*" if same else "."
        print(seq_a[i] + " " + row)

a = "GATTACAGATTACA"
b = "GATTTCAGATACA"
print("window = 1")
dot_plot(a, b, window=1)
print("\nwindow = 3")
dot_plot(a, b, window=3)

### 예시 2 — Needleman–Wunsch 전역 정렬

In [ ]:
import numpy as np
import pandas as pd

def needleman_wunsch(seq1, seq2, match=1, mismatch=-1, gap=-2):
    n, m = len(seq1), len(seq2)
    F = np.zeros((n + 1, m + 1), dtype=int)
    F[:, 0] = gap * np.arange(n + 1)          # 첫 열: seq1을 전부 gap과 짝지은 경우
    F[0, :] = gap * np.arange(m + 1)          # 첫 행: seq2를 전부 gap과 짝지은 경우

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match if seq1[i - 1] == seq2[j - 1] else mismatch
            F[i, j] = max(F[i - 1, j - 1] + s,   # 대각선: 두 글자를 짝짓는다
                          F[i - 1, j] + gap,     # 위: seq2 쪽에 gap
                          F[i, j - 1] + gap)     # 왼쪽: seq1 쪽에 gap

    out1, out2 = "", ""                        # 오른쪽 아래에서 거꾸로 되짚기(traceback)
    i, j = n, m
    while i > 0 or j > 0:
        s = match if i > 0 and j > 0 and seq1[i - 1] == seq2[j - 1] else mismatch
        if i > 0 and j > 0 and F[i, j] == F[i - 1, j - 1] + s:
            out1, out2 = seq1[i - 1] + out1, seq2[j - 1] + out2
            i, j = i - 1, j - 1
        elif i > 0 and F[i, j] == F[i - 1, j] + gap:
            out1, out2 = seq1[i - 1] + out1, "-" + out2
            i = i - 1
        else:
            out1, out2 = "-" + out1, seq2[j - 1] + out2
            j = j - 1
    return F, out1, out2

seq1, seq2 = "GATTACA", "GCTACA"
F, a1, a2 = needleman_wunsch(seq1, seq2)
print(pd.DataFrame(F, index=list("-" + seq1), columns=list("-" + seq2)))
print()
print(a1)
print(a2)
print("점수:", F[-1, -1])

# 자기 점검: 정렬 결과의 점수를 직접 더하면 표의 마지막 칸과 같아야 한다
check = sum(-2 if "-" in (x, y) else (1 if x == y else -1) for x, y in zip(a1, a2))
assert check == F[-1, -1]

### 예시 3 — Smith–Waterman 국소 정렬

In [ ]:
import numpy as np

def smith_waterman(seq1, seq2, match=2, mismatch=-1, gap=-2):
    n, m = len(seq1), len(seq2)
    H = np.zeros((n + 1, m + 1), dtype=int)   # 첫 행·첫 열은 0으로 둔다
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match if seq1[i - 1] == seq2[j - 1] else mismatch
            H[i, j] = max(0,                    # 음수가 되면 0에서 새로 시작
                          H[i - 1, j - 1] + s,
                          H[i - 1, j] + gap,
                          H[i, j - 1] + gap)

    i, j = np.unravel_index(H.argmax(), H.shape)   # 가장 큰 칸에서 출발
    best = H[i, j]
    end1, end2 = i, j
    out1, out2 = "", ""
    while H[i, j] > 0:                             # 0을 만나면 멈춘다
        s = match if seq1[i - 1] == seq2[j - 1] else mismatch
        if H[i, j] == H[i - 1, j - 1] + s:
            out1, out2 = seq1[i - 1] + out1, seq2[j - 1] + out2
            i, j = i - 1, j - 1
        elif H[i, j] == H[i - 1, j] + gap:
            out1, out2 = seq1[i - 1] + out1, "-" + out2
            i = i - 1
        else:
            out1, out2 = "-" + out1, seq2[j - 1] + out2
            j = j - 1
    return best, out1, out2, (i + 1, end1), (j + 1, end2)

long_seq = "MPSLDAKTKEGVVHGLWEQ"      # 가상 단백질
short_seq = "RRKTKQGVVHAA"            # 가운데에만 비슷한 구간이 있는 가상 조각
score, a1, a2, pos1, pos2 = smith_waterman(long_seq, short_seq)
print("최고 점수:", score)
print(f"seq1 {pos1[0]:>2}  {a1}  {pos1[1]}")
print(f"seq2 {pos2[0]:>2}  {a2}  {pos2[1]}")

## 직접 해보기

3. **gap 벌점 실험.** 예시 2를 `gap=0`과 `gap=-5`로 각각 실행해 정렬이 어떻게 달라지는지 봅니다. 예시의 자기 점검 줄은 gap이 −2라고 가정하므로, 아래처럼 함수만 따로 호출합니다.

In [ ]:
for g in [0, -2, -5]:
    F, a1, a2 = needleman_wunsch("GATTACA", "GCTACA", gap=g)
    print(f"gap {g:>2}: {a1} / {a2}  점수 {F[-1, -1]}")